# Notebook 20 — Synthetic Controls and Generalization Tests

**Locked template mode:** same folder/export structure as Notebooks 16–19.

This notebook tests whether the low-rank two-step memory signal from Notebooks 16–19 is a reproducible residue-transition structure rather than a sampling artifact. It compares the real prime-residue transition sequence against multiple controls:

- iid residue shuffle
- first-order Markov synthetic sequence
- block shuffle preserving local chunks
- residue-balanced shuffle preserving global counts
- gap-size shuffle preserving the prime-gap multiset while breaking order

The main object remains

\[
\Delta = P^{(2)}_{\mathrm{emp}} - P^2,
\]

where \(P\) is the first-order residue transition operator and \(P^{(2)}_{\mathrm{emp}}\) is the empirical two-step operator.

In [ ]:
# ============================================================
# Notebook 20 — Synthetic Controls and Generalization Tests
# Locked template cell: imports, IDs, folders
# ============================================================

import os
import math
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "font.size": 12,
})

NOTEBOOK_ID = "20_synthetic_controls_generalization"
OUTDIR = Path(NOTEBOOK_ID)
FIGDIR = OUTDIR / "figures"
DATADIR = OUTDIR / "data"
DOCDIR = OUTDIR / "docs"
TEXDIR = OUTDIR / "tex"

for d in [OUTDIR, FIGDIR, DATADIR, DOCDIR, TEXDIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG_SEED = 9423
rng = np.random.default_rng(RNG_SEED)

print("Notebook ID:", NOTEBOOK_ID)
print("Output directory:", OUTDIR.resolve())

## 1. Prime generation and residue-state encoding

We use the eight reduced residue classes modulo 30:

\[
\mathcal{R}_{30}=\{1,7,11,13,17,19,23,29\}.
\]

Each prime \(p_n > 5\) maps to one of these eight states. This state sequence is the input for all real and synthetic transition operators.

In [ ]:
# ============================================================
# Prime generation + residue state sequence
# ============================================================

MAX_N = 2_000_000
RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
RES_TO_STATE = {int(r): i for i, r in enumerate(RESIDUES)}
N_STATES = len(RESIDUES)


def sieve_primes(n: int) -> np.ndarray:
    """Return all primes <= n using a compact NumPy sieve."""
    if n < 2:
        return np.array([], dtype=int)
    is_prime = np.ones(n + 1, dtype=bool)
    is_prime[:2] = False
    is_prime[4::2] = False
    limit = int(math.isqrt(n))
    for p in range(3, limit + 1, 2):
        if is_prime[p]:
            is_prime[p*p::2*p] = False
    return np.flatnonzero(is_prime)


primes = sieve_primes(MAX_N)
primes = primes[primes > 5]
residue_values = primes % 30
states = np.array([RES_TO_STATE[int(r)] for r in residue_values], dtype=int)
gaps = np.diff(primes)

print("Number of primes > 5:", len(primes))
print("Number of state transitions:", len(states) - 1)
print("Residues:", RESIDUES.tolist())
print("First 12 states:", states[:12].tolist())

pd.DataFrame({
    "max_n": [MAX_N],
    "num_primes_gt5": [len(primes)],
    "num_states": [len(states)],
    "num_transitions": [len(states) - 1],
    "num_gaps": [len(gaps)],
    "residues_mod30": [" ".join(map(str, RESIDUES))],
}).to_csv(DATADIR / "20_dataset_summary.csv", index=False)

## 2. Operator helpers and memory metrics

For each sequence, we estimate:

\[
P_{ij}=\Pr(s_{n+1}=j\mid s_n=i),
\]

\[
P^{(2)}_{ij}=\Pr(s_{n+2}=j\mid s_n=i),
\]

and the two-step memory residual:

\[
\Delta = P^{(2)}_{\mathrm{emp}} - P^2.
\]

The control comparison uses L2 residual, Jensen-Shannon divergence, singular spectrum, rank requirement, entropy rate, and mutual information excess.

In [ ]:
# ============================================================
# Operator, entropy, JS, and residual helpers
# ============================================================

EPS = 1e-12


def row_normalize(M: np.ndarray, eps: float = EPS) -> np.ndarray:
    M = np.asarray(M, dtype=float)
    row_sums = M.sum(axis=1, keepdims=True)
    return np.divide(M, row_sums, out=np.zeros_like(M), where=row_sums > eps)


def transition_operator(seq: np.ndarray, lag: int = 1, n_states: int = N_STATES) -> np.ndarray:
    counts = np.zeros((n_states, n_states), dtype=float)
    for a, b in zip(seq[:-lag], seq[lag:]):
        counts[int(a), int(b)] += 1.0
    return row_normalize(counts)


def stationary_from_counts(seq: np.ndarray, n_states: int = N_STATES) -> np.ndarray:
    counts = np.bincount(seq, minlength=n_states).astype(float)
    return counts / max(counts.sum(), EPS)


def entropy_rate(P: np.ndarray, pi: np.ndarray | None = None) -> float:
    if pi is None:
        pi = np.ones(P.shape[0]) / P.shape[0]
    P_safe = np.clip(P, EPS, 1.0)
    H_rows = -(P_safe * np.log2(P_safe)).sum(axis=1)
    return float(np.dot(pi, H_rows))


def mutual_information_lag(seq: np.ndarray, lag: int = 1, n_states: int = N_STATES) -> float:
    joint = np.zeros((n_states, n_states), dtype=float)
    for a, b in zip(seq[:-lag], seq[lag:]):
        joint[int(a), int(b)] += 1.0
    joint /= max(joint.sum(), EPS)
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    expected = px @ py
    mask = joint > 0
    return float(np.sum(joint[mask] * np.log2(joint[mask] / np.clip(expected[mask], EPS, None))))


def js_divergence(P: np.ndarray, Q: np.ndarray) -> float:
    p = np.asarray(P, dtype=float).ravel()
    q = np.asarray(Q, dtype=float).ravel()
    p = np.clip(p, EPS, None); q = np.clip(q, EPS, None)
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log2(p / m))
    kl_qm = np.sum(q * np.log2(q / m))
    return float(0.5 * (kl_pm + kl_qm))


def rank_for_energy(s: np.ndarray, threshold: float) -> int:
    energy = np.cumsum(s**2) / max(np.sum(s**2), EPS)
    return int(np.searchsorted(energy, threshold) + 1)


def metrics_for_sequence(seq: np.ndarray, name: str) -> tuple[dict, np.ndarray, np.ndarray, np.ndarray]:
    P = transition_operator(seq, lag=1)
    P2_emp = transition_operator(seq, lag=2)
    P2_markov = row_normalize(P @ P)
    delta = P2_emp - P2_markov
    svals = np.linalg.svd(delta, compute_uv=False)
    pi = stationary_from_counts(seq)
    mi1 = mutual_information_lag(seq, lag=1)
    # shuffle expectation from a quick internal permutation baseline
    perm = rng.permutation(seq)
    mi1_shuffle = mutual_information_lag(perm, lag=1)
    metric = {
        "control": name,
        "two_step_l2_residual": float(np.linalg.norm(delta, ord="fro")),
        "two_step_l1_residual": float(np.sum(np.abs(delta))),
        "js_empirical_vs_markov": js_divergence(P2_emp, P2_markov),
        "top_singular_value": float(svals[0]),
        "rank90": rank_for_energy(svals, 0.90),
        "rank95": rank_for_energy(svals, 0.95),
        "entropy_rate_bits": entropy_rate(P, pi),
        "mutual_information_lag1_bits": mi1,
        "mi_excess_over_shuffle_bits": float(mi1 - mi1_shuffle),
        "sequence_length": int(len(seq)),
    }
    return metric, P, P2_emp, delta


def low_rank_delta(delta: np.ndarray, k: int) -> np.ndarray:
    U, s, Vt = np.linalg.svd(delta, full_matrices=False)
    return (U[:, :k] * s[:k]) @ Vt[:k, :]


def corrected_two_step(P: np.ndarray, delta: np.ndarray, k: int) -> np.ndarray:
    Q = P @ P + low_rank_delta(delta, k)
    Q = np.clip(Q, 0, None)
    return row_normalize(Q)

## 3. Synthetic controls

The controls preserve different amounts of information:

1. **iid shuffle:** breaks order while preserving all states.
2. **Markov synthetic:** preserves first-order transition operator \(P\), but removes explicit higher-order memory.
3. **Block shuffle:** preserves short local chunks but randomizes chunk ordering.
4. **Residue-balanced shuffle:** preserves global residue counts.
5. **Gap-size shuffle:** preserves the observed prime-gap multiset but breaks gap ordering before reconstructing residues.

In [ ]:
# ============================================================
# Synthetic controls
# ============================================================

CONTROL_NAMES = [
    "real_prime_sequence",
    "iid_residue_shuffle",
    "markov_synthetic_from_P",
    "block_shuffle_len_8",
    "residue_balanced_shuffle",
    "gap_size_shuffle",
]


def simulate_markov(P: np.ndarray, length: int, start_state: int | None = None) -> np.ndarray:
    if start_state is None:
        start_state = int(rng.integers(0, P.shape[0]))
    out = np.empty(length, dtype=int)
    out[0] = start_state
    for i in range(1, length):
        probs = P[out[i-1]]
        if probs.sum() <= EPS:
            probs = np.ones(P.shape[0]) / P.shape[0]
        out[i] = int(rng.choice(P.shape[0], p=probs / probs.sum()))
    return out


def block_shuffle(seq: np.ndarray, block_len: int = 8) -> np.ndarray:
    n_blocks = int(np.ceil(len(seq) / block_len))
    blocks = [seq[i*block_len:(i+1)*block_len] for i in range(n_blocks)]
    order = rng.permutation(len(blocks))
    return np.concatenate([blocks[i] for i in order])[:len(seq)]


def residue_balanced_shuffle(seq: np.ndarray) -> np.ndarray:
    counts = np.bincount(seq, minlength=N_STATES)
    balanced = np.concatenate([np.full(c, i, dtype=int) for i, c in enumerate(counts)])
    return rng.permutation(balanced)


def gap_size_shuffle_states(primes: np.ndarray, n_states: int = N_STATES) -> np.ndarray:
    # Preserve the observed gap multiset, shuffle gap order, reconstruct a pseudo-prime path,
    # and remap valid mod30 residues. Invalid residue steps are projected to nearest observed residue state.
    shuffled_gaps = rng.permutation(np.diff(primes))
    pseudo = np.empty(len(primes), dtype=np.int64)
    pseudo[0] = int(primes[0])
    pseudo[1:] = pseudo[0] + np.cumsum(shuffled_gaps)
    residues = pseudo % 30
    mapped = np.empty(len(residues), dtype=int)
    for i, r in enumerate(residues):
        if int(r) in RES_TO_STATE:
            mapped[i] = RES_TO_STATE[int(r)]
        else:
            # deterministic projection to closest reduced residue class on circular mod30 distance
            distances = np.minimum((RESIDUES - r) % 30, (r - RESIDUES) % 30)
            mapped[i] = int(np.argmin(distances))
    return mapped

P_real = transition_operator(states, lag=1)

controls = {
    "real_prime_sequence": states.copy(),
    "iid_residue_shuffle": rng.permutation(states),
    "markov_synthetic_from_P": simulate_markov(P_real, len(states), start_state=int(states[0])),
    "block_shuffle_len_8": block_shuffle(states, block_len=8),
    "residue_balanced_shuffle": residue_balanced_shuffle(states),
    "gap_size_shuffle": gap_size_shuffle_states(primes),
}

for name, seq in controls.items():
    print(name, len(seq), np.bincount(seq, minlength=N_STATES).tolist())

## 4. Metric table: real sequence versus controls

The primary test is whether the real sequence has a larger and more structured two-step residual than the controls.

In [ ]:
# ============================================================
# Compute metrics for all controls
# ============================================================

metrics = []
operators = {}
deltas = {}
singular_rows = []
rank_rows = []

for name, seq in controls.items():
    metric, P, P2_emp, delta = metrics_for_sequence(seq, name)
    metrics.append(metric)
    operators[name] = {"P": P, "P2_emp": P2_emp, "P2_markov": row_normalize(P @ P)}
    deltas[name] = delta
    svals = np.linalg.svd(delta, compute_uv=False)
    total_energy = np.sum(svals**2)
    cumulative = np.cumsum(svals**2) / max(total_energy, EPS)
    for k, (s, e) in enumerate(zip(svals, cumulative), start=1):
        singular_rows.append({
            "control": name,
            "rank": k,
            "singular_value": float(s),
            "cumulative_energy": float(e),
        })
    rank_rows.append({
        "control": name,
        "rank90": metric["rank90"],
        "rank95": metric["rank95"],
    })

metrics_df = pd.DataFrame(metrics)
singular_df = pd.DataFrame(singular_rows)
rank_df = pd.DataFrame(rank_rows)

metrics_df.to_csv(DATADIR / "20_control_metrics.csv", index=False)
singular_df.to_csv(DATADIR / "20_singular_spectrum_controls.csv", index=False)
rank_df.to_csv(DATADIR / "20_rank_requirement_controls.csv", index=False)

metrics_df

## 5. Main comparison figures

The core comparisons are residual magnitude, JS divergence, rank requirement, singular spectrum, entropy rate, and mutual information excess.

In [ ]:
# ============================================================
# Figure helpers
# ============================================================

CONTROL_LABELS = {
    "real_prime_sequence": "real",
    "iid_residue_shuffle": "iid shuffle",
    "markov_synthetic_from_P": "Markov synthetic",
    "block_shuffle_len_8": "block shuffle",
    "residue_balanced_shuffle": "balanced shuffle",
    "gap_size_shuffle": "gap shuffle",
}

order = CONTROL_NAMES
x = np.arange(len(order))
labels = [CONTROL_LABELS[o] for o in order]


def savefig(name: str):
    path = FIGDIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved", path)

In [ ]:
# Figure: two-step L2 residual comparison
plt.figure(figsize=(11, 6))
y = [metrics_df.set_index("control").loc[o, "two_step_l2_residual"] for o in order]
plt.bar(x, y)
plt.xticks(x, labels, rotation=25, ha="right")
plt.ylabel("Frobenius norm of Δ")
plt.title("Two-step residual L2: real sequence versus controls")
savefig("20_control_residual_l2_comparison.png")

In [ ]:
# Figure: JS divergence comparison
plt.figure(figsize=(11, 6))
y = [metrics_df.set_index("control").loc[o, "js_empirical_vs_markov"] for o in order]
plt.bar(x, y)
plt.xticks(x, labels, rotation=25, ha="right")
plt.ylabel("JS divergence")
plt.title("Empirical two-step operator distance from Markov baseline")
savefig("20_control_js_comparison.png")

In [ ]:
# Figure: rank requirement comparison
plt.figure(figsize=(11, 6))
width = 0.36
rank90 = [rank_df.set_index("control").loc[o, "rank90"] for o in order]
rank95 = [rank_df.set_index("control").loc[o, "rank95"] for o in order]
plt.bar(x - width/2, rank90, width=width, label="rank for 90% energy")
plt.bar(x + width/2, rank95, width=width, label="rank for 95% energy")
plt.xticks(x, labels, rotation=25, ha="right")
plt.ylabel("rank")
plt.title("Residual energy rank requirement across controls")
plt.legend()
savefig("20_control_energy_rank_comparison.png")

In [ ]:
# Figure: singular spectrum comparison
plt.figure(figsize=(11, 6))
for name in order:
    sdf = singular_df[singular_df["control"] == name]
    plt.plot(sdf["rank"], sdf["singular_value"], marker="o", label=CONTROL_LABELS[name])
plt.xlabel("singular rank")
plt.ylabel("singular value of Δ")
plt.title("Real versus control singular spectra")
plt.legend()
savefig("20_real_vs_controls_singular_spectrum.png")

In [ ]:
# Figure: entropy rate comparison
plt.figure(figsize=(11, 6))
y = [metrics_df.set_index("control").loc[o, "entropy_rate_bits"] for o in order]
plt.bar(x, y)
plt.xticks(x, labels, rotation=25, ha="right")
plt.ylabel("entropy rate (bits)")
plt.title("First-order entropy rate across controls")
savefig("20_entropy_rate_control_comparison.png")

In [ ]:
# Figure: mutual information excess comparison
plt.figure(figsize=(11, 6))
y = [metrics_df.set_index("control").loc[o, "mi_excess_over_shuffle_bits"] for o in order]
plt.axhline(0, linestyle="--")
plt.bar(x, y)
plt.xticks(x, labels, rotation=25, ha="right")
plt.ylabel("MI excess over internal shuffle (bits)")
plt.title("Mutual information excess across controls")
savefig("20_mutual_information_control_comparison.png")

In [ ]:
# Figure: compact metric summary using normalized bars
summary_cols = [
    "two_step_l2_residual",
    "js_empirical_vs_markov",
    "top_singular_value",
    "mi_excess_over_shuffle_bits",
]
summary = metrics_df.set_index("control").loc[order, summary_cols].copy()
summary_norm = summary.copy()
for col in summary_cols:
    v = summary[col].to_numpy(dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    summary_norm[col] = (v - lo) / max(hi - lo, EPS)

summary_norm.to_csv(DATADIR / "20_normalized_control_metric_summary.csv")

plt.figure(figsize=(12, 6))
width = 0.18
for i, col in enumerate(summary_cols):
    plt.bar(x + (i - 1.5) * width, summary_norm[col], width=width, label=col)
plt.xticks(x, labels, rotation=25, ha="right")
plt.ylabel("normalized score")
plt.title("Control metric summary")
plt.legend(fontsize=9)
savefig("20_control_metric_radar_or_bar_summary.png")

## 6. Delta heatmaps

These heatmaps compare the real memory residual \(\Delta\) against the main control residuals. If the real signal is meaningful, it should show a distinct structured pattern instead of control-like noise.

In [ ]:
# Figure: real versus controls delta heatmaps
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
vmax = max(np.max(np.abs(deltas[name])) for name in order)
for ax, name in zip(axes, order):
    im = ax.imshow(deltas[name], aspect="auto", vmin=-vmax, vmax=vmax, cmap="coolwarm")
    ax.set_title(CONTROL_LABELS[name])
    ax.set_xticks(range(N_STATES)); ax.set_xticklabels(RESIDUES)
    ax.set_yticks(range(N_STATES)); ax.set_yticklabels(RESIDUES)
    ax.set_xlabel("state after two steps")
    ax.set_ylabel("current state")
fig.colorbar(im, ax=axes.tolist(), shrink=0.85, label="Δ probability")
plt.suptitle("Two-step residual Δ heatmaps: real versus controls", y=1.02)
plt.tight_layout()
path = FIGDIR / "20_real_vs_controls_delta_heatmaps.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
print("saved", path)

## 7. Low-rank correction validation against controls

For each sequence, we compare the Markov two-step prediction \(P^2\) against corrected models \(P^2 + M_k\), where \(M_k\) is the rank-\(k\) truncation of \(\Delta\).

In [ ]:
# ============================================================
# Low-rank correction validation
# ============================================================

correction_rows = []
for name in order:
    P = operators[name]["P"]
    P2_emp = operators[name]["P2_emp"]
    delta = deltas[name]
    base_err = np.linalg.norm(P2_emp - row_normalize(P @ P), ord="fro")
    for k in range(0, N_STATES + 1):
        if k == 0:
            Q = row_normalize(P @ P)
        else:
            Q = corrected_two_step(P, delta, k)
        err = np.linalg.norm(P2_emp - Q, ord="fro")
        correction_rows.append({
            "control": name,
            "rank_k": k,
            "l2_error": float(err),
            "relative_error_vs_markov": float(err / max(base_err, EPS)),
        })

correction_df = pd.DataFrame(correction_rows)
correction_df.to_csv(DATADIR / "20_low_rank_correction_errors.csv", index=False)

plt.figure(figsize=(11, 6))
for name in order:
    cdf = correction_df[correction_df["control"] == name]
    plt.plot(cdf["rank_k"], cdf["relative_error_vs_markov"], marker="o", label=CONTROL_LABELS[name])
plt.xlabel("rank k")
plt.ylabel("relative L2 error vs Markov")
plt.title("Low-rank correction effectiveness across controls")
plt.legend()
savefig("20_low_rank_correction_control_comparison.png")

## 8. Interpretation summary

This section turns metrics into simple paper-ready comparisons.

In [ ]:
# ============================================================
# Interpretation summary
# ============================================================

m = metrics_df.set_index("control")
real = m.loc["real_prime_sequence"]
control_only = m.drop(index="real_prime_sequence")

summary_rows = []
for metric in [
    "two_step_l2_residual",
    "js_empirical_vs_markov",
    "top_singular_value",
    "entropy_rate_bits",
    "mutual_information_lag1_bits",
    "mi_excess_over_shuffle_bits",
]:
    vals = control_only[metric]
    summary_rows.append({
        "metric": metric,
        "real_value": float(real[metric]),
        "control_mean": float(vals.mean()),
        "control_std": float(vals.std(ddof=0)),
        "real_minus_control_mean": float(real[metric] - vals.mean()),
        "real_over_control_mean": float(real[metric] / max(vals.mean(), EPS)),
    })

interpretation_summary = pd.DataFrame(summary_rows)
interpretation_summary.to_csv(DATADIR / "20_interpretation_summary.csv", index=False)
interpretation_summary

## 9. Paper-ready docs and TeX

This creates markdown and TeX snippets using the locked output folders.

In [ ]:
# ============================================================
# Paper-ready docs and TeX
# ============================================================

real_l2 = float(real["two_step_l2_residual"])
ctrl_l2 = float(control_only["two_step_l2_residual"].mean())
real_js = float(real["js_empirical_vs_markov"])
ctrl_js = float(control_only["js_empirical_vs_markov"].mean())
real_top_s = float(real["top_singular_value"])
ctrl_top_s = float(control_only["top_singular_value"].mean())
real_rank90 = int(real["rank90"])
real_rank95 = int(real["rank95"])

md_text = f"""# Notebook 20 interpretation — Synthetic controls and generalization

Notebook 20 compares the real prime-residue sequence against iid, Markov, block-shuffle, residue-balanced, and gap-shuffle controls.

## Main quantitative readings

- Real two-step L2 residual: `{real_l2:.6f}`
- Mean control two-step L2 residual: `{ctrl_l2:.6f}`
- Real JS divergence from Markov baseline: `{real_js:.6f}`
- Mean control JS divergence: `{ctrl_js:.6f}`
- Real top singular value: `{real_top_s:.6f}`
- Mean control top singular value: `{ctrl_top_s:.6f}`
- Real rank for 90% residual energy: `{real_rank90}`
- Real rank for 95% residual energy: `{real_rank95}`

## Interpretation

The real sequence is tested against controls that preserve different amounts of residue information. A persistent low-rank residual in the real sequence, especially when distinguished from iid and Markov controls, supports the interpretation that the two-step deviation is structured rather than a simple sampling artifact.
"""

(DOCDIR / "20_interpretation.md").write_text(md_text)

tex_text = rf"""
\subsection{{Synthetic Controls and Generalization}}

To test whether the observed two-step residual is a sampling artifact, we compare the real prime-residue sequence against iid, Markov, block-shuffle, residue-balanced, and gap-shuffle controls. For each sequence, we estimate
\[
\Delta = P^{{(2)}}_{{\mathrm{{emp}}}} - P^2,
\]
where $P$ is the first-order transition operator and $P^{{(2)}}_{{\mathrm{{emp}}}}$ is the empirical two-step transition operator.

For the real sequence, the Frobenius norm of $\Delta$ is {real_l2:.6f}, compared with a mean control value of {ctrl_l2:.6f}. The Jensen--Shannon divergence between $P^{{(2)}}_{{\mathrm{{emp}}}}$ and $P^2$ is {real_js:.6f}, compared with a mean control value of {ctrl_js:.6f}. The top singular value of the real residual is {real_top_s:.6f}, and the residual reaches 90\% and 95\% cumulative energy at ranks {real_rank90} and {real_rank95}, respectively.

These controls support the interpretation that the two-step memory residual is not merely an iid or first-order Markov artifact. Instead, it behaves as a structured low-rank transition signal.
"""

(TEXDIR / "20_synthetic_controls_generalization.tex").write_text(tex_text)

print(md_text)
print("Wrote:", DOCDIR / "20_interpretation.md")
print("Wrote:", TEXDIR / "20_synthetic_controls_generalization.tex")

## 10. Locked-template manifest and export zip

The export zip uses the same structure as Notebooks 16–19:

```text
20_synthetic_controls_generalization/
  figures/
  data/
  docs/
  tex/
```

and creates:

```text
20_synthetic_controls_generalization_export.zip
```

In [ ]:
# ============================================================
# Manifest + locked-template export zip
# ============================================================

manifest_rows = []
for subdir in [FIGDIR, DATADIR, DOCDIR, TEXDIR]:
    for path in sorted(subdir.glob("*")):
        if path.is_file():
            manifest_rows.append({
                "notebook_id": NOTEBOOK_ID,
                "relative_path": str(path),
                "folder": path.parent.name,
                "filename": path.name,
                "size_bytes": path.stat().st_size,
            })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(DATADIR / "20_outputs_manifest.csv", index=False)

EXPORT_ZIP = Path(f"{NOTEBOOK_ID}_export.zip")

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTDIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=str(path))

print("Export zip created:", EXPORT_ZIP)
print("Files in manifest:", len(manifest))
print("Zip size bytes:", EXPORT_ZIP.stat().st_size)

# Optional: download outputs bundle (template standard)
# from google.colab import files
# files.download(f"{NOTEBOOK_ID}_export.zip")